# Project 2 — Teach a Neural Network to Read Your Handwriting

### AI Builders Lab · Class 2 · August 23

Project 1 predicted a **number**. This one picks a **category** — which of the ten digits
0–9 is in this picture?

That is the second of the two shapes almost every AI problem takes:

| | **Regression** (Project 1) | **Classification** (Project 2) |
|---|---|---|
| Question | *how much?* | *which one?* |
| Output layer | 1 neuron, no activation | 10 neurons, softmax |
| Loss function | mean squared error | cross-entropy |
| Score you report | mean absolute error | accuracy % |

Everything else — load, look, clean, split, build, train, predict — is **exactly the same
seven steps you just did.** Watch for that. The skeleton never changes.

At the end, the model reads a digit **you** draw with your mouse.

---
Run cells with **Shift + Enter**, in order, top to bottom.

---
# Part 1 — Open the toolbox

Same bench, one new drawer: `mnist`, the dataset itself, which ships inside Keras.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import mnist

tf.keras.utils.set_random_seed(42)

print("TensorFlow version:", tf.__version__)
print("Toolbox is open.")

---
# Part 2 — Get the data

**MNIST** is 70,000 handwritten digits collected from US Census Bureau employees and
American high school students in the 1990s. Every image is 28 x 28 pixels, greyscale,
and comes with a label saying which digit a human says it is.

It is the "Hello World" of deep learning. Every person who has ever trained a neural
network has trained one on this.

Notice that the split into training and testing is **already done for us** here — MNIST
ships as 60,000 training images and 10,000 test images. In Project 1 we had to do that split
ourselves with `train_test_split`. Same idea, done by somebody else.

In [ ]:
(X_train, Y_train), (X_test, Y_test) = mnist.load_data()

# X = the images (the questions).   Y = the correct digit (the answers).
print("Training images:", X_train.shape)   # (60000, 28, 28) -> 60000 images, each 28x28 pixels
print("Training labels:", Y_train.shape)   # (60000,)
print("Testing images: ", X_test.shape)    # (10000, 28, 28) -> the sealed exam
print("Testing labels: ", Y_test.shape)

---
# Part 3 — Look at the data before you model it

Same rule as Project 1. **Never train on data you have not looked at.**

In Project 1 "looking" meant reading a table. Here it means looking at pictures.

In [ ]:
plt.figure(figsize=(12, 3))
for i in range(10):
    plt.subplot(1, 10, i + 1)
    plt.imshow(X_train[i], cmap='gray')   # cmap='gray' - these are greyscale, not colour
    plt.title(Y_train[i])                 # the correct answer, printed on top
    plt.axis('off')
plt.suptitle("The first 10 training images, with their labels")
plt.show()

### The important part: a picture *is* a table of numbers

To a computer there is no such thing as an image. There is only a grid of numbers, one per
pixel, from **0 (black)** to **255 (white)**.

So this is the same kind of data as Project 1 after all. There, one sample was a row of
5 numbers. Here, one sample is a grid of 28 x 28 = **784 numbers**. Same idea, wider row.

Run the cell below and squint at it — you can read the digit straight out of the text.

In [ ]:
# Print image 0 as text: '#' for bright ink, '+' for faint, '.' for empty paper.
print("This image is labelled:", Y_train[0], "\n")
for row in X_train[0]:
    print("".join("#" if p > 128 else ("+" if p > 40 else ".") for p in row))

---
# Part 4 — Prepare the data (normalization)

Our pixels run 0 to 255. We squeeze them into **0.0 to 1.0** by dividing by 255.

### Why bother?

A network learns by nudging its internal weights in small steps. If the inputs are huge
(255) the nudges become huge, the model overshoots, and training gets unstable — you see
the loss bounce around instead of settling.

Keeping every input in roughly the same small range is called **normalization**, and it is
one of the most reliable "my model won't train" fixes you will ever apply.

**One rule you must never break: whatever you do to the training data, do to everything
else.** Divide the training images by 255 but forget the test images, and your test score
collapses for a reason that looks like a modelling failure but is really a bookkeeping error.
We will come back to this rule at the very end of the notebook, and it will matter.

In [ ]:
X_train = X_train / 255.0
X_test  = X_test  / 255.0

print("Smallest pixel value now:", X_train.min())
print("Largest pixel value now: ", X_train.max())

---
# Part 5 — Build the network

Look at the diagram on screen. Here is the architecture and the reason for each layer:

| Layer | Size | Why |
|---|---|---|
| `Input` | 28 x 28 | The shape of one image. |
| `Flatten` | 784 | A Dense layer wants a single row of numbers, not a square. So we unroll the grid. **We are throwing away the information about which pixel is next to which** — remember this; it is exactly the weakness that CNNs will fix in a later module. |
| `Dense` + ReLU | 128 | Hidden layer 1. Learns strokes and fragments. |
| `Dropout` | 0.2 | Randomly ignore 20% of neurons while training. Anti-overfitting, same as Project 1. |
| `Dense` + ReLU | 64 | Hidden layer 2. Combines fragments into shapes. |
| `Dense` + **softmax** | **10** | One neuron per digit. Softmax turns them into ten probabilities that add to 100%. |

### Softmax — the one genuinely new idea here

Project 1 ended with one bare neuron because we wanted a raw number.

Here we want a *decision between ten options*, so we end with ten neurons and pass them
through **softmax**. Softmax takes ten arbitrary numbers, exaggerates the differences by
taking `exp` of each, then divides by the total so they sum to exactly 1.

Example: `[2.0, 1.0, 0.1]` becomes `[0.66, 0.24, 0.10]`.

The model therefore never says "it is a 7." It says "I am 94% sure it is a 7, 3% sure it is a
1, ..." — and we take the biggest. **That confidence number is real information.** A model
that is 40% sure is telling you something a model that is 99% sure is not.

In [ ]:
model = models.Sequential([
    layers.Input(shape=(28, 28)),              # tell the model what one image looks like
    layers.Flatten(),                          # 28x28 square -> 784 numbers in a row
    layers.Dense(128, activation='relu'),      # hidden layer 1
    layers.Dropout(0.2),                       # randomly ignore 20% of neurons while training
    layers.Dense(64,  activation='relu'),      # hidden layer 2
    layers.Dense(10,  activation='softmax')    # 10 outputs = 10 digits, as probabilities
])

model.summary()

### Read the summary — the parameter count is the point

About **109,000 numbers** the network will adjust while learning. Nobody sets those by hand.

Check the first Dense layer: 100,480 parameters.

$$784 \text{ inputs} \times 128 \text{ neurons} = 100{,}352 \text{ weights}, \quad +\ 128 \text{ biases} = 100{,}480$$

Same formula as Project 1: `(in x out) + out`. Project 1's network had 6,951 parameters;
this one has 109,386. The problem is harder, so the brain is bigger.

---
# Part 6 — Compile: set the rules of learning

Three settings, and notice how two of them differ from Project 1:

**`optimizer='adam'` — HOW to learn.** Same as Project 1. Big steps early when it is badly
wrong, small careful steps later. The sensible default for almost everything.

**`loss='sparse_categorical_crossentropy'` — HOW WRONG is it.** This is the change.
Project 1 used mean squared error, which measures distance between numbers. That is
meaningless here — digit 8 is not "two more wrong" than digit 6. Cross-entropy instead
punishes the model for the confidence it put on the wrong answers. Being 99% sure and wrong
costs far more than being 30% sure and wrong. *"Sparse"* just means our labels are plain
integers (`7`) rather than lists of ten zeros and a one.

**`metrics=['accuracy']` — the number WE read.** The loss is for the optimizer; accuracy is
for us. "What fraction did it get right.

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("Model compiled. The rules of learning are set.")

---
# Part 7 — Train it

- **`epochs=5`** — five full passes over all 60,000 images.
- **`batch_size=128`** — update the weights after every 128 images.
- **`validation_data=(X_test, Y_test)`** — check against the sealed exam after each pass,
  so we can watch for overfitting as it happens.

This takes about a minute on CPU. **Watch the `accuracy` column climb.** After the very
first epoch it is already reading handwriting better than a random guess by a factor of ten.

In [ ]:
history = model.fit(
    X_train, Y_train,
    epochs=5,
    batch_size=128,
    validation_data=(X_test, Y_test),
    verbose=1
)

print("\nTraining finished.")

---
# Part 8 — How well did it do?

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test, Y_test, verbose=0)

print("ACCURACY on unseen data:", round(test_accuracy * 100, 2), "%")
print("\nOut of 10,000 digits it had never seen, it got about",
      int(test_accuracy * 10000), "right.")

You should be around **97–98%**.

Sit with that for a second. We wrote about six lines of model code. The program was never
told a single rule about what digits look like — no "a 7 has a horizontal bar", no "an 8
has two loops". It worked all of it out from examples alone, in under a minute.

That is the thing that is genuinely new about machine learning, and it is worth being
slightly amazed by before it becomes normal to you.

In [ ]:
plt.figure(figsize=(11, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='training')
plt.plot(history.history['val_accuracy'], label='unseen test data')
plt.title('Accuracy (higher is better)'); plt.xlabel('Epoch'); plt.legend(loc='lower right')
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='training')
plt.plot(history.history['val_loss'], label='unseen test data')
plt.title('Loss (lower is better)'); plt.xlabel('Epoch'); plt.legend(loc='upper right')
plt.grid(alpha=0.3)

plt.tight_layout(); plt.show()

### Reading these graphs — the same skill as Project 1

Watch the **gap** between the two lines.

- Close together → learning a real pattern. Good.
- Training line pulling far above the test line → **overfitting**, it is memorising.
- Test loss turning upward while training loss keeps falling → trained too long.

Every model you ever train produces this pair of curves, and you read them the same way
every time.

### One prediction, in detail

Softmax gives ten probabilities, not one answer. Let us look at all ten.

In [ ]:
predictions = model.predict(X_test, verbose=0)

i = 9        # CHANGE THIS NUMBER and re-run to inspect a different test image

probabilities = predictions[i]
guess = int(np.argmax(probabilities))     # argmax = "which position holds the biggest number?"

plt.figure(figsize=(9, 3.2))
plt.subplot(1, 2, 1)
plt.imshow(X_test[i], cmap='gray'); plt.axis('off')
plt.title(f"true answer: {Y_test[i]}   model says: {guess}")
plt.subplot(1, 2, 2)
plt.bar(range(10), probabilities)
plt.xticks(range(10)); plt.ylim(0, 1)
plt.title("how sure it is about each digit")
plt.tight_layout(); plt.show()

print("Confidence in its answer:", round(float(probabilities[guess]) * 100, 2), "%")

---
# Part 9 — Save the model

One line, and the 109,386 learned numbers are on disk. Real models take days to train —
you never retrain something you already have.

In [ ]:
model.save('handwriting_model.keras')
print("Saved to handwriting_model.keras")

# Click the FOLDER icon in Colab's left sidebar to see the file.
# Colab wipes this storage when the session ends - right-click -> Download to keep it.

---
---
# Part 10 — THE FUN PART: make it read *your* handwriting

Everything so far used somebody else's data. Now we point the model at you.

There is a problem to solve first, and it is the most useful practical lesson in this notebook.

### The model is extremely fussy about its input

It has only ever seen images that are:

- exactly **28 x 28** pixels
- **white ink on a black background** — not black ink on white paper
- **centred**, and sized to fill about 20 of the 28 pixels
- with pixel values between **0 and 1**

Your drawing will be none of those things.

Feed it in raw and the model will confidently return garbage. **It will not raise an error.
It will just be wrong.** That silence is exactly what makes this failure mode dangerous —
and it is the same rule from Part 4: *whatever you did to the training data, do to your
own data too.*

So we write a function that converts any picture of a digit into MNIST's exact format.
Read the six steps in the comments — step 5, centring on the centre of mass, is the one
everybody skips and the number one reason "it works on MNIST but not on my handwriting".

In [ ]:
from PIL import Image

def prepare_digit(img, show=True):
    """Convert ANY picture of a single digit into the 28x28 format the model expects."""

    original = img
    g = np.array(img.convert("L"), dtype=np.float32)   # "L" = convert to greyscale

    # STEP 1: Flip black-on-white to white-on-black.
    # MNIST is white ink on black paper. Your notebook paper is the opposite.
    # We check the brightness of the border pixels: if the edge of the picture is
    # light, we are looking at paper, so we invert.
    border = np.concatenate([g[0, :], g[-1, :], g[:, 0], g[:, -1]])
    if border.mean() > 127:
        g = 255.0 - g

    # STEP 2: Stretch the contrast and delete the faint stuff.
    # Photos have shadows, paper texture, and grey smudges. Anything dimmer than
    # 40% of the brightest ink is treated as background and set to pure black.
    g = g - g.min()
    if g.max() > 0:
        g = g / g.max() * 255.0
    g[g < 0.4 * g.max()] = 0

    # STEP 3: Crop away the empty space, keeping only the ink.
    ys, xs = np.nonzero(g)
    if len(ys) == 0:
        print("I can't find any ink in this image. Try drawing darker or thicker.")
        return np.zeros((28, 28), dtype=np.float32)
    g = g[ys.min():ys.max() + 1, xs.min():xs.max() + 1]

    # STEP 4: Resize so the longest side is 20 pixels.
    # Why 20 and not 28? Because the people who built MNIST scaled every digit into
    # a 20x20 box and left a 4-pixel margin. We copy them exactly. Get this wrong
    # and your digit is the wrong size compared to everything the model studied.
    h, w = g.shape
    scale = 20.0 / max(h, w)
    new_h, new_w = max(1, int(round(h * scale))), max(1, int(round(w * scale)))
    g = np.array(Image.fromarray(g.astype(np.uint8)).resize((new_w, new_h), Image.LANCZOS),
                 dtype=np.float32)

    # STEP 5: Paste into a 28x28 black square, centred by CENTRE OF MASS.
    # MNIST centres each digit on its centre of gravity, not its bounding box.
    # This is the step everyone skips, and skipping it is the #1 reason
    # "it works on MNIST but not on my own writing".
    canvas = np.zeros((28, 28), dtype=np.float32)
    top, left = (28 - new_h) // 2, (28 - new_w) // 2
    canvas[top:top + new_h, left:left + new_w] = g
    cy, cx = np.array(np.nonzero(canvas)).mean(axis=1)
    canvas = np.roll(canvas, int(round(13.5 - cy)), axis=0)
    canvas = np.roll(canvas, int(round(13.5 - cx)), axis=1)

    # STEP 6: Scale to 0-1, exactly as we did to the training data.
    canvas = canvas / 255.0

    if show:
        plt.figure(figsize=(7, 3))
        plt.subplot(1, 2, 1); plt.imshow(original, cmap='gray')
        plt.title("What you gave it"); plt.axis('off')
        plt.subplot(1, 2, 2); plt.imshow(canvas, cmap='gray')
        plt.title("What the model actually sees"); plt.axis('off')
        plt.show()

    return canvas


def predict_digit(img28):
    """Feed a prepared 28x28 image to the model and show the verdict."""
    p = model.predict(img28.reshape(1, 28, 28), verbose=0)[0]
    guess = int(np.argmax(p))

    print("=" * 44)
    print("   THE MODEL SAYS:", guess, "  (", round(p[guess] * 100, 1), "% confident )")
    print("=" * 44)
    print("\nIts full opinion:")
    for digit in range(10):
        bar = "#" * int(p[digit] * 40)
        print(f"  {digit} | {bar:<40} {p[digit]*100:5.1f}%")
    return guess

print("Helper functions ready.")

## Draw a digit with your mouse

Run the cell below. A black box appears. **Draw one digit in it** by holding the mouse
button down and dragging, then click **DONE**.

Tips for a good result:
- Draw **big** — fill most of the box.
- Draw **thick and confidently**. Thin scratchy lines vanish when we shrink to 28x28.
- One digit only. Click **Clear** to start over.

In [ ]:
from IPython.display import HTML, display
from google.colab.output import eval_js
from base64 import b64decode
import io

canvas_html = '''
<div style="font-family: sans-serif;">
  <canvas id="pad" width="280" height="280"
          style="border:3px solid #555; background:#000; cursor:crosshair; touch-action:none;"></canvas>
  <br><br>
  <button id="done"  style="font-size:16px; padding:8px 18px; cursor:pointer;">DONE - read my digit</button>
  <button id="clear" style="font-size:16px; padding:8px 18px; cursor:pointer;">Clear</button>
</div>
<script>
  var c   = document.getElementById('pad');
  var ctx = c.getContext('2d');
  ctx.fillStyle = 'black';
  ctx.fillRect(0, 0, c.width, c.height);
  ctx.strokeStyle = 'white';
  ctx.lineWidth   = 20;          // thick, so the stroke survives shrinking to 28x28
  ctx.lineCap     = 'round';
  ctx.lineJoin    = 'round';

  var drawing = false;
  function spot(e) {
    var r = c.getBoundingClientRect();
    return [e.clientX - r.left, e.clientY - r.top];
  }
  c.addEventListener('pointerdown', function(e) {
    drawing = true;
    var p = spot(e);
    ctx.beginPath();
    ctx.moveTo(p[0], p[1]);
    ctx.lineTo(p[0], p[1]);
    ctx.stroke();
  });
  c.addEventListener('pointermove', function(e) {
    if (!drawing) return;
    var p = spot(e);
    ctx.lineTo(p[0], p[1]);
    ctx.stroke();
  });
  window.addEventListener('pointerup', function() { drawing = false; });

  document.getElementById('clear').onclick = function() {
    ctx.fillStyle = 'black';
    ctx.fillRect(0, 0, c.width, c.height);
  };

  // Python waits on this promise until you click DONE
  var data = new Promise(function(resolve) {
    document.getElementById('done').onclick = function() {
      resolve(c.toDataURL('image/png'));
    };
  });
</script>
'''

display(HTML(canvas_html))
drawing_data = eval_js("data")                       # pauses here until you click DONE
raw = b64decode(drawing_data.split(',')[1])
my_drawing = Image.open(io.BytesIO(raw))

print("Got your drawing.\n")

In [ ]:
# Now clean it up and ask the model
prepared = prepare_digit(my_drawing)
predict_digit(prepared)

**Did it get it right?**

Go back, draw a different digit, run both cells again, and **try to find one it fails on.**

When it fails, look at the right-hand picture — *what the model actually sees*. Nine times
out of ten the failure is visible right there: the stroke got too thin, the digit sat in a
corner, or you drew a shape genuinely unlike MNIST's American 1990s style (a European 7 with
a crossbar, a 1 with a flag and a base serif, an open-topped 4).

That is not a bug. That is the model honestly telling you it has never seen handwriting like
yours. **A model can only recognise what it has been shown.** Remember that sentence — it
explains most of the AI failures you will read about in the news.

---
# What you just did, and what to notice

Compare the two projects side by side:

| Step | Project 1 (crime) | Project 2 (digits) |
|---|---|---|
| 1. Load data | `pd.read_csv` from GitHub | `mnist.load_data()` |
| 2. Look at it | rows = samples, columns = features | pictures, then the same picture as numbers |
| 3. Clean | drop impossible negatives | divide by 255 |
| 4. Split | `train_test_split`, 80/20 | already split for us |
| 5. Build | 5 → 100 → 50 → 25 → **1** | 784 → 128 → 64 → **10 softmax** |
| 6. Train | `model.fit` | `model.fit` |
| 7. Predict | one invented city | one digit you drew |

**Two totally different problems. One skeleton.** Learn the skeleton and you can attack
anything — that is what makes the rest of this course possible.

### Homework

1. Write all ten digits 0–9 by hand and test each one. Which does it get wrong? Look at
   "what the model actually sees" and write down *why* you think it failed.
2. Change **one thing** in the model — `Dense(128)` → `Dense(16)`, or delete the `Dropout`,
   or set `epochs=1` — re-run, and record what happened to the test accuracy. One change at
   a time. If you change three things at once you learn nothing about any of them.
3. The full self-study version of this notebook is in the class repository
   (`03_Handwriting_FULL_selfstudy.ipynb`). It goes further: it shows you the digits the
   model gets wrong, and lets you upload a photo of real pen-on-paper handwriting.